In [ ]:
!pip install -q accelerate

In [ ]:
%%writefile /kaggle/working/train_ddp_accelerate.py
# PRETRAINING CONDIVISO PER MODEL MERGING (theta0)
# ==================================================
# Obiettivo: ottenere un unico set di pesi theta0, ottenuto da inizializzazione
# random, allenato su un pretext task GENERICO di "conditional restoration",
# da usare come punto di partenza comune per il fine-tuning separato dei tre
# task (star removal, image restoration, super-resolution) prima del merging.
#
# Differenze rispetto allo script di fine-tuning single-task:
#   1) Pesi iniziali random: non si cerca ne' si carica alcun theta0 esterno.
#   2) TARGET = unione delle immagini pulite prese dai TRAIN split dei tre
#      task, ricavati dai rispettivi split_manifest.json (mai da val/test).
#   3) CONDITION = le stesse immagini target, degradate con una pipeline di
#      degradazioni SINTETICHE random (rumore, blur, downsample/upsample,
#      occlusioni puntiformi), scelte in modo indipendente dal task di
#      provenienza dell'immagine, cosi' theta0 non impara alcuna
#      associazione implicita tra contenuto e tipo di corruzione.
#
# Architettura (SelfAttention, UNetBlock, SinusoidalPositionEmbeddings,
# ImprovedUNet, DDPMvPrediction, EMA) INVARIATA rispetto allo script originale.

import os
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
import math
import json
import torch
import torch.nn as nn
import torch.optim as optim
import random
import glob
import numpy as np
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.utils import make_grid
import matplotlib.pyplot as plt
from tqdm import tqdm
import copy

from accelerate import Accelerator
from accelerate.utils import set_seed as accelerate_set_seed

# =========================================================================
# ARCHITETTURA INVARIATA (SelfAttention, UNetBlock, SinusoidalPositionEmbeddings, ImprovedUNet)
# =========================================================================
class SelfAttention(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.channels = channels
        self.mha = nn.MultiheadAttention(embed_dim=channels, num_heads=2, batch_first=True)
        self.ln = nn.LayerNorm(channels)
        self.ff_self = nn.Sequential(
            nn.LayerNorm(channels),
            nn.Linear(channels, channels),
            nn.GELU(),
            nn.Linear(channels, channels),
        )

    def forward(self, x):
        size = x.shape[-1]
        x_flat = x.reshape(-1, self.channels, size * size).transpose(1, 2)
        x_ln = self.ln(x_flat)
        attention_value, _ = self.mha(x_ln, x_ln, x_ln)
        attention_value = attention_value + x_flat
        attention_value = self.ff_self(attention_value) + attention_value
        return attention_value.transpose(1, 2).reshape(-1, self.channels, size, size)


class UNetBlock(nn.Module):
    def __init__(self, in_channels, out_channels, time_dim=32, num_groups=4):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1)
        self.gn1 = nn.GroupNorm(num_groups, out_channels)
        self.act1 = nn.SiLU()

        self.time_mlp = nn.Linear(time_dim, out_channels)

        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.gn2 = nn.GroupNorm(num_groups, out_channels)
        self.act2 = nn.SiLU()

        self.residual_conv = nn.Conv2d(in_channels, out_channels, kernel_size=1) if in_channels != out_channels else nn.Identity()

    def forward(self, x, t_emb):
        h = self.act1(self.gn1(self.conv1(x)))
        time_proj = self.time_mlp(t_emb).unsqueeze(-1).unsqueeze(-1)
        h = h + time_proj
        h = self.act2(self.gn2(self.conv2(h)))
        return h + self.residual_conv(x)


class SinusoidalPositionEmbeddings(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, time):
        device = time.device
        half_dim = self.dim // 2
        embeddings = math.log(10000) / (half_dim - 1)
        embeddings = torch.exp(torch.arange(half_dim, device=device) * -embeddings)
        embeddings = time[:, None] * embeddings[None, :]
        embeddings = torch.cat((embeddings.sin(), embeddings.cos()), dim=-1)
        return embeddings


class ImprovedUNet(nn.Module):
    def __init__(self, in_channels=3, cond_channels=3, base_channels=64):
        super().__init__()

        time_dim = base_channels * 4
        self.time_mlp = nn.Sequential(
            SinusoidalPositionEmbeddings(base_channels),
            nn.Linear(base_channels, time_dim),
            nn.GELU(),
            nn.Linear(time_dim, time_dim)
        )

        c = base_channels

        self.inc = UNetBlock(in_channels + cond_channels, c, time_dim)

        self.down1 = nn.Conv2d(c, c, kernel_size=4, stride=2, padding=1)
        self.enc1 = UNetBlock(c, c * 2, time_dim)

        self.down2 = nn.Conv2d(c * 2, c * 2, kernel_size=4, stride=2, padding=1)
        self.enc2 = UNetBlock(c * 2, c * 4, time_dim)

        self.down3 = nn.Conv2d(c * 4, c * 4, kernel_size=4, stride=2, padding=1)
        self.enc3 = UNetBlock(c * 4, c * 8, time_dim)
        self.attn3 = SelfAttention(c * 8)

        self.down4 = nn.Conv2d(c * 8, c * 8, kernel_size=4, stride=2, padding=1)

        self.bottleneck1 = UNetBlock(c * 8, c * 8, time_dim)
        self.attention = SelfAttention(c * 8)
        self.bottleneck2 = UNetBlock(c * 8, c * 8, time_dim)

        self.up1 = nn.ConvTranspose2d(c * 8, c * 8, kernel_size=4, stride=2, padding=1)
        self.dec1 = UNetBlock(c * 16, c * 4, time_dim)
        self.attn_up1 = SelfAttention(c * 4)

        self.up2 = nn.ConvTranspose2d(c * 4, c * 4, kernel_size=4, stride=2, padding=1)
        self.dec2 = UNetBlock(c * 8, c * 2, time_dim)

        self.up3 = nn.ConvTranspose2d(c * 2, c * 2, kernel_size=4, stride=2, padding=1)
        self.dec3 = UNetBlock(c * 4, c, time_dim)

        self.up4 = nn.ConvTranspose2d(c, c, kernel_size=4, stride=2, padding=1)
        self.dec4 = UNetBlock(c * 2, c, time_dim)

        self.out = nn.Conv2d(c, in_channels, kernel_size=3, padding=1)

    def forward(self, x_t, t, condition):
        x_input = torch.cat([x_t, condition], dim=1)
        t_emb = self.time_mlp(t.float())

        s1 = self.inc(x_input, t_emb)
        h = self.down1(s1)

        s2 = self.enc1(h, t_emb)
        h = self.down2(s2)

        s3 = self.enc2(h, t_emb)
        h = self.down3(s3)

        s4 = self.enc3(h, t_emb)
        s4 = self.attn3(s4)
        h = self.down4(s4)

        h = self.bottleneck1(h, t_emb)
        h = self.attention(h)
        h = self.bottleneck2(h, t_emb)

        h = self.up1(h)
        h = torch.cat([h, s4], dim=1)
        h = self.dec1(h, t_emb)
        h = self.attn_up1(h)

        h = self.up2(h)
        h = torch.cat([h, s3], dim=1)
        h = self.dec2(h, t_emb)

        h = self.up3(h)
        h = torch.cat([h, s2], dim=1)
        h = self.dec3(h, t_emb)

        h = self.up4(h)
        h = torch.cat([h, s1], dim=1)
        h = self.dec4(h, t_emb)

        return self.out(h)


# =========================================================================
def cosine_beta_schedule(timesteps, s=0.008):
    steps = timesteps + 1
    x = torch.linspace(0, timesteps, steps)
    alphas_cumprod = torch.cos(((x / timesteps) + s) / (1 + s) * math.pi * 0.5) ** 2
    alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
    betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
    return torch.clip(betas, 1e-4, 0.9999)


class DDPMvPrediction(nn.Module):
    def __init__(self, network, n_steps=300, beta_start=1e-4, beta_end=0.02):
        super().__init__()
        self.network = network
        self.n_steps = n_steps

        beta = cosine_beta_schedule(n_steps)
        alpha = 1.0 - beta
        alpha_bar = torch.cumprod(alpha, dim=0)
        alpha_bar_prev = torch.cat([torch.tensor([1.0]), alpha_bar[:-1]])

        sqrt_alpha_bar = torch.sqrt(alpha_bar)
        sqrt_one_minus_alpha_bar = torch.sqrt(1.0 - alpha_bar)

        self.register_buffer('alpha_bar', alpha_bar)
        self.register_buffer('alpha_bar_prev', alpha_bar_prev)
        self.register_buffer('sqrt_alpha_bar', sqrt_alpha_bar)
        self.register_buffer('sqrt_one_minus_alpha_bar', sqrt_one_minus_alpha_bar)
        self.register_buffer('beta', beta)
        self.register_buffer('alpha', alpha)
        self.register_buffer('posterior_variance', beta * (1 - alpha_bar_prev) / (1 - alpha_bar))

    def forward_diffusion(self, x_0, t, noise):
        sqrt_alpha_bar_t = self.sqrt_alpha_bar[t].view(-1, 1, 1, 1)
        sqrt_one_minus_alpha_bar_t = self.sqrt_one_minus_alpha_bar[t].view(-1, 1, 1, 1)
        return sqrt_alpha_bar_t * x_0 + sqrt_one_minus_alpha_bar_t * noise

    def forward(self, x_0, condition, t=None, noise=None):
        batch_size = x_0.shape[0]
        if t is None:
            t = torch.randint(0, self.n_steps, (batch_size,), device=x_0.device)
        if noise is None:
            noise = torch.randn_like(x_0)

        x_t = self.forward_diffusion(x_0, t, noise)

        sqrt_alpha_bar_t = self.sqrt_alpha_bar[t].view(-1, 1, 1, 1)
        sqrt_one_minus_alpha_bar_t = self.sqrt_one_minus_alpha_bar[t].view(-1, 1, 1, 1)

        v_target = sqrt_alpha_bar_t * noise - sqrt_one_minus_alpha_bar_t * x_0
        predicted_v = self.network(x_t, t.float(), condition)

        loss = F.mse_loss(predicted_v, v_target)
        return loss

    @torch.no_grad()
    def sample(self, condition):
        self.network.eval()
        device = self.beta.device

        batch_size, channels, height, width = condition.shape
        x = torch.randn(batch_size, channels, height, width, device=device)

        for i in reversed(range(self.n_steps)):
            t = torch.full((batch_size,), i, dtype=torch.long, device=device)
            predicted_v = self.network(x, t.float(), condition)

            sqrt_alpha_bar_t = self.sqrt_alpha_bar[i]
            sqrt_one_minus_alpha_bar_t = self.sqrt_one_minus_alpha_bar[i]

            pred_x0 = (sqrt_alpha_bar_t * x) - (sqrt_one_minus_alpha_bar_t * predicted_v)
            pred_x0 = torch.clamp(pred_x0, -1.0, 1.0)

            beta_t = self.beta[i]
            alpha_bar_prev_t = self.alpha_bar_prev[i]
            alpha_bar_t = self.alpha_bar[i]
            alpha_t = self.alpha[i]

            mean = (beta_t * torch.sqrt(alpha_bar_prev_t) / (1.0 - alpha_bar_t)) * pred_x0 + \
                   ((1.0 - alpha_bar_prev_t) * torch.sqrt(alpha_t) / (1.0 - alpha_bar_t)) * x

            if i > 0:
                noise = torch.randn_like(x)
                sigma_t = torch.sqrt(self.posterior_variance[i])
                x = mean + sigma_t * noise
            else:
                x = mean

        self.network.train()
        return x


def strip_compile_prefix(state_dict):
    if not state_dict:
        return state_dict
    if all(k.startswith("_orig_mod.") for k in state_dict.keys()):
        return {k[len("_orig_mod."):]: v for k, v in state_dict.items()}
    return state_dict


class ExponentialMovingAverage:
    def __init__(self, beta=0.999):
        self.beta = beta

    def update(self, ema_model, current_model):
        with torch.no_grad():
            for ema_param, current_param in zip(ema_model.parameters(), current_model.parameters()):
                ema_param.data.mul_(self.beta).add_(current_param.data, alpha=1.0 - self.beta)


# =========================================================================
# DATASET DI PRETRAINING: target pooled dai 3 task, condizione = degradazione
# SINTETICA random (NON la degradazione reale del task specifico)
# =========================================================================
class SyntheticDegradationDataset(Dataset):
    """
    - target: caricato dai .npy indicati in target_paths (unione dei train
      split dei tre task, ricavati dai rispettivi split_manifest.json)
    - condition: la STESSA immagine, degradata on-the-fly ad ogni accesso con
      una combinazione random di rumore / blur / downsample-upsample /
      occlusioni sintetiche puntiformi. La scelta della degradazione NON
      dipende in alcun modo da quale task ha fornito quell'immagine: questo
      e' il punto chiave per evitare che theta0 impari un'associazione
      implicita tra contenuto e tipo di corruzione.
    - p_cond_dropout: probabilita' di azzerare completamente la condizione
      (classifier-free guidance style), per non far perdere al modello la
      capacita' di generare anche con guida debole/assente.
    """

    def __init__(self, target_paths, severity_range=(0.05, 0.35), p_cond_dropout=0.1):
        self.target_paths = list(target_paths)
        assert len(self.target_paths) > 0, "La lista di target path e' vuota."
        self.severity_range = severity_range
        self.p_cond_dropout = p_cond_dropout

    def __len__(self):
        return len(self.target_paths)

    def _load_clean(self, path):
        arr = np.load(path).astype(np.float32)
        if arr.max() > 1.5:  # eventuali npy salvati in [0,255]
            arr = arr / 255.0
        if arr.ndim == 2:
            arr = arr[..., None]
        return arr  # (H, W, C) in [0,1]

    # ---- singole degradazioni sintetiche (operano su tensori CHW in [0,1]) ----
    def _add_noise(self, x):
        sigma = random.uniform(*self.severity_range)
        if random.random() < 0.5:
            x = x + torch.randn_like(x) * sigma
        else:
            scale = 1.0 / max(sigma, 1e-3)
            x = torch.poisson(torch.clamp(x, 0, 1) * scale) / scale
        return torch.clamp(x, 0.0, 1.0)

    def _blur(self, x):
        k = random.choice([3, 5, 7, 9])
        sigma = random.uniform(0.5, 2.5)
        c = x.shape[0]
        coords = torch.arange(k, dtype=torch.float32) - k // 2
        g = torch.exp(-(coords ** 2) / (2 * sigma ** 2))
        g = (g / g.sum())
        kernel_x = g.view(1, 1, 1, k).expand(c, 1, 1, k)
        kernel_y = g.view(1, 1, k, 1).expand(c, 1, k, 1)
        x = x.unsqueeze(0)
        x = F.conv2d(x, kernel_x, padding=(0, k // 2), groups=c)
        x = F.conv2d(x, kernel_y, padding=(k // 2, 0), groups=c)
        return x.squeeze(0).clamp(0.0, 1.0)

    def _down_up_sample(self, x):
        scale = random.choice([2, 3, 4])
        c, h, w = x.shape
        x = x.unsqueeze(0)
        x_down = F.interpolate(x, size=(max(h // scale, 4), max(w // scale, 4)),
                                mode="bicubic", align_corners=False)
        x_up = F.interpolate(x_down, size=(h, w), mode="bicubic", align_corners=False)
        return x_up.squeeze(0).clamp(0.0, 1.0)

    def _synthetic_occlusions(self, x):
        c, h, w = x.shape
        n_holes = random.randint(1, 6)
        x = x.clone()
        for _ in range(n_holes):
            rh = random.randint(max(h // 40, 2), max(h // 10, 3))
            rw = random.randint(max(w // 40, 2), max(w // 10, 3))
            cy = random.randint(0, h - 1)
            cx = random.randint(0, w - 1)
            y0, y1 = max(cy - rh, 0), min(cy + rh, h)
            x0, x1 = max(cx - rw, 0), min(cx + rw, w)
            local_mean = x[:, y0:y1, x0:x1].mean(dim=(1, 2), keepdim=True)
            fill = torch.clamp(local_mean + (torch.rand(c, 1, 1) - 0.5) * 0.2, 0.0, 1.0)
            x[:, y0:y1, x0:x1] = fill
        return x

    def _degrade(self, target_chw):
        x = target_chw.clone()
        pool = [self._add_noise, self._blur, self._down_up_sample, self._synthetic_occlusions]
        random.shuffle(pool)
        n_ops = random.randint(1, 3)
        for op in pool[:n_ops]:
            x = op(x)
        return torch.clamp(x, 0.0, 1.0)

    def __getitem__(self, idx):
        target_arr = self._load_clean(self.target_paths[idx])
        target_tensor = torch.from_numpy(target_arr).float()
        if target_tensor.shape[-1] in [1, 3]:
            target_tensor = target_tensor.permute(2, 0, 1)  # CHW

        if random.random() < self.p_cond_dropout:
            condition_tensor = torch.zeros_like(target_tensor)
        else:
            condition_tensor = self._degrade(target_tensor)

        condition_tensor = (condition_tensor * 2.0) - 1.0
        target_tensor = (target_tensor * 2.0) - 1.0

        return condition_tensor, target_tensor


@torch.no_grad()
def compute_stable_val_loss(ddpm_ema, val_dataloader, device, n_mc_samples=4, seed=123):
    """
    Loss di validazione a bassa varianza: per ogni batch valuta la loss su
    n_mc_samples combinazioni di (t, rumore) FISSE e riproducibili (stesso
    seed ogni epoca), invece di un singolo draw casuale come nel forward()
    standard. Questo elimina la componente di rumore dovuta al campionamento
    stocastico di t/noise, che altrimenti puo' far sembrare "migliore" (o
    "peggiore") un'epoca solo per fortuna, non per una reale differenza di
    qualita' dei pesi.
    """
    total_loss = 0.0
    n_batches = 0
    for batch in val_dataloader:
        condition, x_0 = batch
        condition, x_0 = condition.to(device), x_0.to(device)

        batch_loss = 0.0
        generator = torch.Generator(device="cpu").manual_seed(seed + n_batches)
        for _ in range(n_mc_samples):
            t = torch.randint(0, ddpm_ema.n_steps, (x_0.shape[0],), generator=generator).to(device)
            noise = torch.randn(x_0.shape, generator=generator).to(device)
            batch_loss += ddpm_ema(x_0, condition, t=t, noise=noise).item()
        total_loss += batch_loss / n_mc_samples
        n_batches += 1

    return total_loss / max(n_batches, 1)


def load_train_targets_from_manifest(manifest_path):
    with open(manifest_path, "r") as f:
        manifest = json.load(f)
    if "train_target" not in manifest:
        raise KeyError(f"Il manifest {manifest_path} non contiene la chiave 'train_target'.")
    return manifest["train_target"]


def build_pooled_target_paths(manifest_paths):
    """Unisce i target dei TRAIN split dei tre task, senza duplicati e senza
    mai toccare val/test (che restano isolati per la valutazione finale)."""
    pooled = []
    seen = set()
    for mp in manifest_paths:
        targets = load_train_targets_from_manifest(mp)
        for t in targets:
            if t not in seen:
                seen.add(t)
                pooled.append(t)
    return pooled


# =========================================================================
# COSTRUZIONE AUTOMATICA DEI MANIFEST DA DATASET GREZZI
# =========================================================================
# Se uno split_manifest atteso non si trova nel dataset di input Kaggle,
# viene ricostruito da zero a partire dalle cartelle grezze dei tre task
# (ciascuna organizzata come <root>/input/npy e <root>/target/npy, con
# file .npy corrispondenti che condividono lo stesso nome). Il manifest
# ricostruito viene salvato in output_dir (scrivibile), MAI nella cartella
# di input Kaggle (read-only), cosi' le run successive lo ritrovano gia'
# pronto senza doverlo ricostruire ogni volta.
def _pair_input_target_files(dataset_root):
    """Ritorna la lista di coppie (input_path, target_path) per un dataset
    organizzato come dataset_root/input/npy/*.npy e dataset_root/target/npy/*.npy,
    accoppiando i file che condividono lo stesso nome."""
    input_dir = os.path.join(dataset_root, "input", "npy")
    target_dir = os.path.join(dataset_root, "target", "npy")

    target_files = sorted(glob.glob(os.path.join(target_dir, "*.npy")))
    assert len(target_files) > 0, f"Nessun file .npy trovato in {target_dir}"

    pairs = []
    for tpath in target_files:
        fname = os.path.basename(tpath)
        ipath = os.path.join(input_dir, fname)
        if os.path.exists(ipath):
            pairs.append((ipath, tpath))

    assert len(pairs) > 0, (f"Nessuna coppia input/target trovata in {dataset_root} "
                             f"(controlla che i nomi file in input/npy e target/npy coincidano).")
    return pairs


def build_manifest_train_val(dataset_root, train_fraction=0.9, seed=42):
    """Split semplice train/val (percentuali su train_fraction) di UN solo
    dataset grezzo organizzato come dataset_root/{input,target}/npy."""
    pairs = _pair_input_target_files(dataset_root)

    rng = random.Random(seed)
    shuffled = pairs.copy()
    rng.shuffle(shuffled)

    n_train = int(round(len(shuffled) * train_fraction))
    train_pairs = shuffled[:n_train]
    val_pairs = shuffled[n_train:]

    return {
        "source_root": dataset_root,
        "train_fraction": train_fraction,
        "train_input": [p[0] for p in train_pairs],
        "train_target": [p[1] for p in train_pairs],
        "val_input": [p[0] for p in val_pairs],
        "val_target": [p[1] for p in val_pairs],
    }


def build_manifest_star_removal(uar_root, normal_root,
                                 uar_train_fraction=0.6, normal_train_fraction=0.3, seed=42):
    """Manifest di star removal, costruito da DUE dataset grezzi distinti:
      - uar_root: se ne usa solo 'uar_train_fraction' (default 60%) per il
        train; il resto (40%) NON viene usato ne' in train ne' in val.
      - normal_root: se ne usa 'normal_train_fraction' (default 30%) per il
        train; tutto il resto va in val.
    """
    uar_pairs = _pair_input_target_files(uar_root)
    normal_pairs = _pair_input_target_files(normal_root)

    rng = random.Random(seed)

    # --- dataset "uar": 60% train, 40% scartato (mai usato) ---
    uar_shuffled = uar_pairs.copy()
    rng.shuffle(uar_shuffled)
    n_uar_train = int(round(len(uar_shuffled) * uar_train_fraction))
    uar_train_pairs = uar_shuffled[:n_uar_train]
    uar_unused_pairs = uar_shuffled[n_uar_train:]  # scartate di proposito

    # --- dataset "normal": 30% train, 70% val ---
    normal_shuffled = normal_pairs.copy()
    rng.shuffle(normal_shuffled)
    n_normal_train = int(round(len(normal_shuffled) * normal_train_fraction))
    normal_train_pairs = normal_shuffled[:n_normal_train]
    normal_val_pairs = normal_shuffled[n_normal_train:]

    train_pairs = uar_train_pairs + normal_train_pairs
    val_pairs = normal_val_pairs  # la parte "uar" non usata NON finisce in val

    return {
        "source_uar_root": uar_root,
        "source_normal_root": normal_root,
        "uar_train_fraction": uar_train_fraction,
        "normal_train_fraction": normal_train_fraction,
        "train_input": [p[0] for p in train_pairs],
        "train_target": [p[1] for p in train_pairs],
        "val_input": [p[0] for p in val_pairs],
        "val_target": [p[1] for p in val_pairs],
        "stats": {
            "uar_total": len(uar_pairs),
            "uar_used_for_train": len(uar_train_pairs),
            "uar_unused": len(uar_unused_pairs),
            "normal_total": len(normal_pairs),
            "normal_used_for_train": len(normal_train_pairs),
            "normal_used_for_val": len(normal_val_pairs),
        },
    }


def get_or_build_manifest(expected_path, fallback_path, builder_fn, task_name, accelerator):
    """Ritorna il path di un manifest json valido per 'task_name':
      1) se esiste gia' come dataset di input Kaggle (expected_path), usa quello;
      2) altrimenti, se e' gia' stato ricostruito in una run precedente
         (fallback_path, dentro output_dir), riusa quello;
      3) altrimenti lo costruisce da zero con builder_fn() e lo salva in
         fallback_path (SOLO il processo main scrive su disco, gli altri
         processi aspettano e poi ricaricano lo stesso file)."""
    if os.path.exists(expected_path):
        if accelerator.is_main_process:
            print(f"Manifest {task_name}: trovato in input locale -> {expected_path}")
        return expected_path

    if os.path.exists(fallback_path):
        if accelerator.is_main_process:
            print(f"Manifest {task_name}: NON trovato in {expected_path}. "
                  f"Riuso quello gia' ricostruito in precedenza -> {fallback_path}")
        return fallback_path

    if accelerator.is_main_process:
        print(f"Manifest {task_name}: NON trovato in {expected_path}. Lo costruisco da zero...")
        manifest_dict = builder_fn()
        os.makedirs(os.path.dirname(fallback_path), exist_ok=True)
        with open(fallback_path, "w") as f:
            json.dump(manifest_dict, f, indent=2)
        print(f"Manifest {task_name}: costruito e salvato in {fallback_path} "
              f"({len(manifest_dict['train_target'])} train / {len(manifest_dict['val_target'])} val)")
    accelerator.wait_for_everyone()

    return fallback_path


# =========================================================================
# TRAINING (DDP tramite accelerate)
# =========================================================================
USE_TORCH_COMPILE = False


def main():
    # =====================================================================
    # CONFIGURAZIONE (stile Kaggle, hardcoded come nel notebook originale).
    # =====================================================================
    accelerator = Accelerator(mixed_precision="fp16")
    accelerate_set_seed(42)
    device = accelerator.device

    torch.backends.cudnn.benchmark = True

    seed = 42  # definito qui perche' serve gia' per costruire (se serve) gli split dei manifest

    output_dir = "/kaggle/working/Risultati_DDPM_Pretrain_theta0"
    if accelerator.is_main_process:
        os.makedirs(output_dir, exist_ok=True)
    accelerator.wait_for_everyone()

    # ---------------------------------------------------------------------
    # Un unico dataset Kaggle di input con tre sottocartelle, una per task,
    # ciascuna con il proprio split_manifest.json generato in fase di split
    # train/val/test del task. MODIFICA "manifests_dataset_root" e i nomi
    # delle sottocartelle in base al tuo dataset Kaggle reale.
    #
    # Se uno di questi manifest non viene trovato, viene ricostruito da zero
    # a partire dai dataset grezzi corrispondenti (vedi builder_fn sotto) e
    # salvato in output_dir/built_manifests (mai nella cartella di input,
    # che su Kaggle e' read-only).
    # ---------------------------------------------------------------------
    manifests_dataset_root = "/kaggle/input/datasets/iladar/split-manifest"

    expected_manifest_star_removal = os.path.join(manifests_dataset_root, "split_manifest.json")
    expected_manifest_restoration = os.path.join(manifests_dataset_root, "split_manifest_IR.json")
    expected_manifest_super_resolution = os.path.join(manifests_dataset_root, "split_manifest_SR.json")

    built_manifests_dir = os.path.join(output_dir, "built_manifests")
    if accelerator.is_main_process:
        os.makedirs(built_manifests_dir, exist_ok=True)
    accelerator.wait_for_everyone()

    fallback_manifest_star_removal = os.path.join(built_manifests_dir, "split_manifest.json")
    fallback_manifest_restoration = os.path.join(built_manifests_dir, "split_manifest_IR.json")
    fallback_manifest_super_resolution = os.path.join(built_manifests_dir, "split_manifest_SR.json")

    # Dataset grezzi Kaggle usati SOLO se il manifest corrispondente non e'
    # gia' disponibile come dataset di input. Ciascuno e' organizzato come
    # <root>/input/npy/*.npy e <root>/target/npy/*.npy con nomi file corrispondenti.
    raw_dataset_root_super_resolution = "/kaggle/input/datasets/phantasm04/super-resolution/dataset_su"
    raw_dataset_root_restoration = "/kaggle/input/datasets/phantasm04/image-restoration/dataset_ir"
    raw_dataset_root_star_removal_uar = "/kaggle/input/datasets/phantasm04/star-removal-uar/dataset"
    raw_dataset_root_star_removal_normal = "/kaggle/input/datasets/phantasm04/star-removal/dataset"

    manifest_super_resolution = get_or_build_manifest(
        expected_manifest_super_resolution,
        fallback_manifest_super_resolution,
        lambda: build_manifest_train_val(raw_dataset_root_super_resolution,
                                          train_fraction=0.8, seed=seed),
        "super_resolution", accelerator,
    )
    manifest_restoration = get_or_build_manifest(
        expected_manifest_restoration,
        fallback_manifest_restoration,
        lambda: build_manifest_train_val(raw_dataset_root_restoration,
                                          train_fraction=0.8, seed=seed),
        "restoration", accelerator,
    )
    manifest_star_removal = get_or_build_manifest(
        expected_manifest_star_removal,
        fallback_manifest_star_removal,
        lambda: build_manifest_star_removal(raw_dataset_root_star_removal_uar,
                                             raw_dataset_root_star_removal_normal,
                                             uar_train_fraction=0.5,
                                             normal_train_fraction=0.3,
                                             seed=seed),
        "star_removal", accelerator,
    )

    monitor_val_fraction = 0.05  # frazione del pool usata SOLO per monitorare la loss di pretraining
    batch_size = 4
    epochs = 1000
    max_epoch_this_session = 150
    n_steps_ddpm = 200
    base_channels = 64
    lr = 1e-4

    # =====================================================================
    # POOL DI PRETRAINING: unione dei target dei TRAIN split dei tre task.
    # I manifest sono quelli generati durante lo split train/val/test di
    # ciascun task; qui usiamo SOLO "train_target" di ciascuno, mai val/test.
    # =====================================================================
    manifest_paths = [manifest_star_removal, manifest_restoration, manifest_super_resolution]

    pool_manifest_path_out = os.path.join(output_dir, "pretrain_pool_manifest.json")
    if os.path.exists(pool_manifest_path_out):
        if accelerator.is_main_process:
            print(f"Trovato pool manifest esistente in: {pool_manifest_path_out}. Lo ricarico per coerenza.")
        with open(pool_manifest_path_out, "r") as f:
            pool_manifest = json.load(f)
        pooled_target_paths = pool_manifest["train_target_pool"]
        monitor_train_paths = pool_manifest["monitor_train"]
        monitor_val_paths = pool_manifest["monitor_val"]
    else:
        pooled_target_paths = build_pooled_target_paths(manifest_paths)
        if accelerator.is_main_process:
            print(f"Pool di pretraining costruito: {len(pooled_target_paths)} immagini target uniche "
                  f"(unione dei train split dei 3 task).")

        rng = random.Random(seed)
        shuffled = pooled_target_paths.copy()
        rng.shuffle(shuffled)
        n_monitor_val = max(1, int(len(shuffled) * monitor_val_fraction))
        monitor_val_paths = shuffled[:n_monitor_val]
        monitor_train_paths = shuffled[n_monitor_val:]

        pool_manifest = {
            "source_manifests": manifest_paths,
            "train_target_pool": pooled_target_paths,
            "monitor_train": monitor_train_paths,
            "monitor_val": monitor_val_paths,
        }
        if accelerator.is_main_process:
            with open(pool_manifest_path_out, "w") as f:
                json.dump(pool_manifest, f, indent=2)
            print(f"Pool manifest salvato in: {pool_manifest_path_out}")

    if accelerator.is_main_process:
        print(f"Pretraining pool: {len(monitor_train_paths)} train (monitor) / {len(monitor_val_paths)} val (monitor)")
    accelerator.wait_for_everyone()

    train_dataset = SyntheticDegradationDataset(monitor_train_paths)
    val_dataset = SyntheticDegradationDataset(monitor_val_paths, p_cond_dropout=0.0)  # niente dropout in val, per una loss di monitoraggio piu' stabile

    train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True,
                                  num_workers=4, pin_memory=True, persistent_workers=True)
    val_dataloader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, drop_last=False,
                                 num_workers=4, pin_memory=True, persistent_workers=True)

    # =====================================================================
    # PESI INIZIALI RANDOM: nessuna ricerca/caricamento di un theta0 esterno,
    # questo script e' esattamente il processo che produce theta0.
    # =====================================================================
    nn_architecture = ImprovedUNet(in_channels=3, base_channels=base_channels)

    init_weights_path = os.path.join(output_dir, "init_random_weights.pth")
    if accelerator.is_main_process and not os.path.exists(init_weights_path):
        torch.save(nn_architecture.state_dict(), init_weights_path)
        print(f"Inizializzazione random salvata (per audit/riproducibilita'): {init_weights_path}")
    accelerator.wait_for_everyone()

    ema_architecture = copy.deepcopy(nn_architecture).to(device)
    ema_architecture.eval()
    ema_updater = ExponentialMovingAverage(beta=0.999)

    ddpm = DDPMvPrediction(nn_architecture, n_steps=n_steps_ddpm)
    ddpm_ema = DDPMvPrediction(ema_architecture, n_steps=n_steps_ddpm).to(device)
    ddpm_ema.eval()

    optimizer = optim.AdamW(ddpm.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    ddpm, optimizer, train_dataloader, val_dataloader, scheduler = accelerator.prepare(
        ddpm, optimizer, train_dataloader, val_dataloader, scheduler
    )

    if USE_TORCH_COMPILE:
        ddpm = torch.compile(ddpm)

    start_epoch = 0
    history_train_loss = []
    history_val_loss = []

    best_val_loss = float('inf')
    best_epoch = -1
    smoothed_val_loss = None  # EMA della curva di val loss, usata per decidere il "best"
    VAL_LOSS_EMA_BETA = 0.85  # piu' alto = piu' smussamento, meno reattivita' a singole epoche fortunate
    MIN_EPOCHS_BEFORE_BEST = 20  # warmup: non salvare "best" prima di questa epoca (loss iniziale troppo instabile)

    # =====================================================================
    # RESUME entro la stessa run (utile per sessioni Kaggle con limite di
    # tempo): cerca solo dentro output_dir, non esiste piu' alcun
    # input_checkpoint_dir esterno perche' non c'e' un theta0 da iniettare.
    # =====================================================================
    all_checkpoints = glob.glob(os.path.join(output_dir, "pretrain_epoch_*.pth"))

    if all_checkpoints:
        all_checkpoints.sort(key=lambda x: int(os.path.basename(x).split('_')[-1].split('.')[0]))
        last_checkpoint = all_checkpoints[-1]

        if accelerator.is_main_process:
            print(f"Trovato checkpoint precedente in: {last_checkpoint}")
        checkpoint = torch.load(last_checkpoint, map_location=device)

        unwrapped_ddpm = accelerator.unwrap_model(ddpm)
        unwrapped_ddpm.load_state_dict(strip_compile_prefix(checkpoint["model"]))
        optimizer.load_state_dict(checkpoint["optimizer"])
        scheduler.load_state_dict(checkpoint["scheduler"])
        start_epoch = checkpoint["epoch"] + 1

        if "ema_model" in checkpoint:
            ema_architecture.load_state_dict(strip_compile_prefix(checkpoint["ema_model"]))
        else:
            if accelerator.is_main_process:
                print("Pesi EMA non trovati, inizializzo EMA dai pesi attuali.")
            ema_architecture.load_state_dict(unwrapped_ddpm.network.state_dict())

        if "history_train_loss" in checkpoint:
            history_train_loss = checkpoint["history_train_loss"]
            history_val_loss = checkpoint.get("history_val_loss", [0] * len(history_train_loss))

        if "best_val_loss" in checkpoint:
            best_val_loss = checkpoint["best_val_loss"]
            best_epoch = checkpoint.get("best_epoch", -1)
        smoothed_val_loss = checkpoint.get("smoothed_val_loss", None)
    else:
        if accelerator.is_main_process:
            print("Inizio il pretraining di theta0 da zero (inizializzazione random).")

    fixed_indices = [0, 1, 2, 3]
    val_condition_list, val_target_list = [], []
    for idx in fixed_indices:
        safe_idx = idx if idx < len(val_dataset) else (idx % len(val_dataset))
        cond, target = val_dataset[safe_idx]
        val_condition_list.append(cond)
        val_target_list.append(target)

    fixed_val_condition_cpu = torch.stack(val_condition_list)
    fixed_val_target_cpu = torch.stack(val_target_list)
    if accelerator.is_main_process:
        print(f"Batch di validazione fisso preparato con gli indici: {fixed_indices}")
        print(f"Inizio del pretraining v-prediction ({epochs} epoche) su {accelerator.num_processes} GPU...")

    for epoch in range(start_epoch, epochs):
        # -- 1. TRAINING --
        ddpm.train()
        total_train_loss = 0.0
        pbar = tqdm(train_dataloader, desc=f"Epoch {epoch+1:03d}/{epochs} [Pretrain]",
                    disable=not accelerator.is_local_main_process, mininterval=15.0)
        for batch in pbar:
            condition, x_0 = batch

            optimizer.zero_grad(set_to_none=True)
            loss = ddpm(x_0, condition)

            accelerator.backward(loss)
            accelerator.clip_grad_norm_(ddpm.parameters(), max_norm=1.0)
            optimizer.step()

            ema_updater.update(ema_architecture, accelerator.unwrap_model(ddpm).network)

            total_train_loss += loss.item()
            pbar.set_postfix(loss=f"{loss.item():.4f}")

        avg_train_loss = total_train_loss / len(train_dataloader)
        history_train_loss.append(avg_train_loss)

        # -- 2. VALIDAZIONE (con modello EMA, deterministica e a bassa varianza) --
        ddpm_ema.eval()
        avg_val_loss = compute_stable_val_loss(ddpm_ema, val_dataloader, device, n_mc_samples=4)
        history_val_loss.append(avg_val_loss)

        # Val loss smussata (EMA sulla CURVA di loss, non sui pesi): serve a
        # decidere il "best" senza reagire a singole epoche fortunate/sfortunate.
        if smoothed_val_loss is None:
            smoothed_val_loss = avg_val_loss
        else:
            smoothed_val_loss = VAL_LOSS_EMA_BETA * smoothed_val_loss + (1 - VAL_LOSS_EMA_BETA) * avg_val_loss

        if accelerator.is_main_process:
            print(f"Epoch {epoch+1:03d} | Train Loss: {avg_train_loss:.4f} | "
                  f"Val Loss (EMA weights, deterministica): {avg_val_loss:.4f} | "
                  f"Val Loss smussata: {smoothed_val_loss:.4f}")

        # === SALVATAGGIO BEST theta0 ===
        # Condizioni: (1) dopo il warmup, (2) sul valore SMUSSATO, non sul
        # valore grezzo dell'epoca corrente -> un'epoca fortunata isolata
        # non basta a spostare il best, serve un miglioramento persistente.
        is_past_warmup = (epoch + 1) >= MIN_EPOCHS_BEFORE_BEST
        if is_past_warmup and smoothed_val_loss < best_val_loss:
            best_val_loss = smoothed_val_loss
            best_epoch = epoch + 1
            if accelerator.is_main_process:
                print(f"👑 Nuova Best Loss di pretraining (smussata): {best_val_loss:.6f} all'epoca {best_epoch}! Salvataggio in corso...")
                unwrapped_ddpm = accelerator.unwrap_model(ddpm)
                torch.save({
                    "epoch": epoch,
                    "model": unwrapped_ddpm.state_dict(),
                    "ema_model": ema_architecture.state_dict(),
                    "optimizer": optimizer.state_dict(),
                    "scheduler": scheduler.state_dict(),
                    "history_train_loss": history_train_loss,
                    "history_val_loss": history_val_loss,
                    "best_val_loss": best_val_loss,
                    "best_epoch": best_epoch,
                    "smoothed_val_loss": smoothed_val_loss,
                }, os.path.join(output_dir, "pretrain_best.pth"))

                # Deliverable pulito da usare come init per i 3 fine-tuning:
                # solo i pesi della U-Net (EMA), senza wrapper ne' buffer di diffusione.
                torch.save(ema_architecture.state_dict(), os.path.join(output_dir, "theta0.pth"))
                print(f"theta0.pth aggiornato (pesi EMA della U-Net, epoca {best_epoch})")

                # Copia di sicurezza extra, con numero di epoca nel nome: utile
                # per confrontare a posteriori piu' "best" successivi invece di
                # fidarsi ciecamente dell'ultimo sovrascritto.
                torch.save(ema_architecture.state_dict(),
                           os.path.join(output_dir, f"theta0_best_epoch_{best_epoch}.pth"))

                KEEP_LAST_N_BEST_SNAPSHOTS = 3
                best_snapshots = sorted(
                    glob.glob(os.path.join(output_dir, "theta0_best_epoch_*.pth")),
                    key=lambda p: int(os.path.basename(p).split('_')[-1].split('.')[0])
                )
                for old_snap in best_snapshots[:-KEEP_LAST_N_BEST_SNAPSHOTS]:
                    try:
                        os.remove(old_snap)
                    except OSError:
                        pass
        # ==============================

        scheduler.step()
        accelerator.wait_for_everyone()

        # -- 3. SALVATAGGIO CICLICO E VISUALIZZAZIONE --
        if (epoch + 1) % 5 == 0 and accelerator.is_main_process:
            print(f"\n---> Generazione test v-prediction all'epoca {epoch+1} (EMA)...")

            unwrapped_ddpm = accelerator.unwrap_model(ddpm)
            torch.save({
                "epoch": epoch,
                "model": unwrapped_ddpm.state_dict(),
                "ema_model": ema_architecture.state_dict(),
                "optimizer": optimizer.state_dict(),
                "scheduler": scheduler.state_dict(),
                "history_train_loss": history_train_loss,
                "history_val_loss": history_val_loss,
                "best_val_loss": best_val_loss,
                "best_epoch": best_epoch
            }, os.path.join(output_dir, f"pretrain_epoch_{epoch+1}.pth"))

            KEEP_LAST_N_CHECKPOINTS = 2
            saved_this_session = sorted(
                glob.glob(os.path.join(output_dir, "pretrain_epoch_*.pth")),
                key=lambda p: int(os.path.basename(p).split('_')[-1].split('.')[0])
            )
            for old_ckpt in saved_this_session[:-KEEP_LAST_N_CHECKPOINTS]:
                try:
                    os.remove(old_ckpt)
                    print(f"Rimosso vecchio checkpoint per liberare spazio: {os.path.basename(old_ckpt)}")
                except OSError as e:
                    print(f"Impossibile rimuovere {old_ckpt}: {e}")

            plt.figure(figsize=(10, 5))
            epoche_range = range(1, len(history_train_loss) + 1)
            plt.plot(epoche_range, history_train_loss, label='Train Loss', color='blue')
            plt.plot(epoche_range, history_val_loss, label='Val Loss (EMA, monitor)', color='orange')
            plt.yscale('log')
            plt.title(f"Andamento Loss di Pretraining (Log Scale) - Epoca {epoch+1}")
            plt.xlabel("Epoche")
            plt.ylabel("MSE Loss (v-prediction)")
            plt.legend()
            plt.grid(True, which="both", ls="--", alpha=0.5)
            plt.tight_layout()
            plt.savefig(os.path.join(output_dir, f"loss_epoch_{epoch+1}.png"))
            plt.close()

            torch.cuda.empty_cache()

            with torch.no_grad():
                val_batch = fixed_val_target_cpu.to(device)
                val_condition = fixed_val_condition_cpu.to(device)
                generated_samples = ddpm_ema.sample(condition=val_condition)

            cond_vis = torch.clamp((val_condition.cpu() + 1.0) / 2.0, 0.0, 1.0)
            gen_vis = torch.clamp((generated_samples.cpu() + 1.0) / 2.0, 0.0, 1.0)
            real_vis = torch.clamp((val_batch.cpu() + 1.0) / 2.0, 0.0, 1.0)

            grid = make_grid(torch.cat([cond_vis, gen_vis, real_vis], dim=0),
                              nrow=fixed_val_condition_cpu.shape[0]).permute(1, 2, 0).numpy()

            plt.figure(figsize=(12, 9))
            plt.imshow(grid)
            plt.title(f"Epoca {epoch+1} v-pred (Indici {fixed_indices}) - EMA\n"
                      f"In alto: Condizione (degradazione sintetica) -> In mezzo: Predizione -> In basso: Target")
            plt.axis('off')
            plt.savefig(os.path.join(output_dir, f"samples_epoch_{epoch+1}.png"))
            plt.close()
            torch.cuda.empty_cache()

        accelerator.wait_for_everyone()

        # -- 4. LIMITE DI SESSIONE --
        if (epoch + 1) >= max_epoch_this_session:
            if accelerator.is_main_process:
                print(f"\nLimite epoche per questa sessione raggiunto ({max_epoch_this_session}). Stop pulito.")
            break

    if accelerator.is_main_process:
        print("\nPretraining di theta0 completato con successo!")
        print(f"theta0 finale disponibile in: {os.path.join(output_dir, 'theta0.pth')}")


if __name__ == "__main__":
    main()

In [ ]:
!accelerate launch --multi_gpu --num_processes=2 --mixed_precision=fp16 /kaggle/working/train_ddp_accelerate.py